<a href="https://colab.research.google.com/github/iAndrey31/CLUSTERING-CNV/blob/main/BIRTH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy

Cargar dataset original

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

ruta_entrada = '/content/drive/MyDrive/IA/BIRTH/birth21-25.csv'
print("Drive montado correctamente.")

Filtrar años 2021-2025

In [ ]:
años_interes = [2021, 2022, 2023, 2024, 2025, '2021', '2022', '2023', '2024', '2025']
ruta_filtrado = '/content/CNV_2021_2025_Filtrado.csv'
chunk_size = 100000
es_primera_vez = True

try:
    for chunk in pd.read_csv(ruta_entrada, chunksize=chunk_size, sep=None,
                             engine='python', on_bad_lines='skip', encoding='utf-8'):
        chunk_filtrado = chunk[chunk['FecNac_Año'].isin(años_interes)]
        if not chunk_filtrado.empty:
            if es_primera_vez:
                chunk_filtrado.to_csv(ruta_filtrado, index=False, mode='w', encoding='utf-8-sig')
                es_primera_vez = False
            else:
                chunk_filtrado.to_csv(ruta_filtrado, index=False, mode='a', header=False, encoding='utf-8-sig')
    print("Filtrado completado.")
except Exception as e:
    print(f"Error: {e}")

Limpieza de valores inválidos

In [ ]:
df = pd.read_csv(ruta_filtrado, sep=None, engine='python',
                 on_bad_lines='skip', encoding='utf-8')

# Eliminar columnas innecesarias
df.drop(columns=['Hijos_vivo_madre', 'Hijos_fallec_madre'], inplace=True, errors='ignore')

# Convertir a numérico y eliminar valores -1
for col in ['PESO_NACIDO', 'TALLA_NACIDO', 'Num_embar_madre']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df[df[col] != -1]

# Filtrar edad mínima
df['Edad_Madre'] = pd.to_numeric(df['Edad_Madre'], errors='coerce')
df = df[df['Edad_Madre'] >= 13]

# Eliminar nulos generados
df.dropna(subset=['PESO_NACIDO', 'TALLA_NACIDO', 'Num_embar_madre', 'Edad_Madre'], inplace=True)

print(f"Registros tras limpieza: {len(df)}")

Renombrar columnas

In [ ]:
mapeo_columnas = {
    'FecNac_Año'            : 'AÑO_NAC',
    'FecNac_Mes'            : 'MES_NAC',
    'DUR_EMB_PARTO'         : 'SEM_EMB',
    'Condicion_Parto'       : 'CONDIC_PARTO',
    'sexo_nacido'           : 'SEXO_NACIDO',
    'Tipo_Parto'            : 'TIPO_PARTO',
    'Edad_Madre'            : 'EDAD_MADRE',
    'Estado_Civil'          : 'ESTADO_CIVIL',
    'Nivel_Intrucción_Madre': 'EDUCACION_MADRE',
    'DESC_OCUPACION'        : 'OCUPACION_MADRE',
    'Num_embar_madre'       : 'EMB_MADRE',
    'nacmuer_abort_madre'   : 'ABORTOS_MADRE',
    'Pais_Madre'            : 'PAIS_MADRE',
    'IdUbigeoInei'          : 'UBIGEO',
    'Ipress'                : 'IPRESS',
    'Lugar_Nacido'          : 'LUGAR_NACIDO',
    'Atiende_Parto'         : 'ATIENDE_PARTO',
    'Financiador_Parto'     : 'FINANCIADOR_PARTO'
}

df.rename(columns=mapeo_columnas, inplace=True)
print("Columnas renombradas:", df.columns.tolist())

Limpiar datos

In [ ]:
df['UBIGEO'] = pd.to_numeric(df['UBIGEO'], errors='coerce')
filas_antes = len(df)
df = df[df['UBIGEO'] != -1].copy()
print(f"Filas eliminadas por UBIGEO=-1: {filas_antes - len(df)}")

# Reemplazar IGNORADO por NO SE MENCIONA
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].replace('IGNORADO', 'NO SE MENCIONA')

print("Limpieza de UBIGEO e IGNORADO completada.")

Clasificar OCUPACION_MADRE

In [ ]:
def clasificar_ocupacion(valor):
    if pd.isna(valor):
        return 'OTRO'
    v = str(valor).upper().strip()

    ama = ['AMA DE CASA', 'AMA DE LLAVES', 'AMA DE CRIA', 'EMPLEADO DEL HOGAR',
           'NANA', 'AYA MAMA', 'CUIDADO DE NIÑOS', 'CUIDADO DE NIÐOS',
           'MUCAMA', 'SIRVIENT', 'MOZO (PERSONAL', 'NIÐERA', 'NINERA']
    if any(p in v for p in ama):
        return 'AMA DE CASA'

    if any(p in v for p in ['PRACTICANTE', 'PASANTE', 'ESTUDIANTE']):
        return 'ESTUDIANTE'

    prof = ['MEDICO', 'ENFERMERA', 'ENFERMERO', 'ODONTOLOGO', 'DENTISTA',
            'FARMACEUT', 'QUIMICO', 'BIOLOGO', 'PSICOLOGO', 'NUTRICION',
            'FISIOTERA', 'TECNOLOGO MEDICO', 'TECNICO DE SALUD', 'AUXILIAR, ENF',
            'AUXILIAR FARM', 'ASISTENTE SOCIAL', 'ASISTENTE MEDICO',
            'TECNICO EN LABORATORIO', 'TECNICO, LABORAT', 'VETERINARIO',
            'INGENIERO', 'ARQUITECTO', 'ECONOMISTA', 'CONTADOR', 'ADMINISTRADOR',
            'ABOGADO', 'JUEZ', 'FISCAL', 'PROFESOR', 'MAESTRO DE ESCUELA',
            'MAESTRO DE ENSEÐ', 'PERIODISTA', 'SOCIOLOGO', 'ANTROPOLOGO',
            'GEOLOGO', 'GEOGRAFO', 'AGRONOMO', 'ZOOTECNICO', 'ESTADISTICO',
            'MATEMATICO', 'ACTUARIO', 'AUDITOR', 'ANALISTA', 'PROGRAMADOR',
            'POLICIA', 'MILITAR', 'MARINA,', 'EJERCITO', 'AVIACION,',
            'PILOTO', 'GERENTE', 'DIRECTOR', 'TECNICO,', 'TECNICOS ',
            'ARQUEOLOG', 'FILOSOFO', 'HISTORIADOR', 'LINGUISTA', 'BIOQU',
            'MICROBIO', 'PEDIATRA', 'GINECOLOGO', 'PSIQUIA', 'DERMATO',
            'OFTALMO', 'CIRUJANO', 'DELINEANTE', 'GEOFISICO', 'AGRIMENSOR',
            'TOPOGRAFO', 'PUBLICISTA', 'RELACIONISTA', 'TRADUCTOR', 'INTERPRETE',
            'ESCRITOR', 'SERENAZGO', 'ECOLOGISTA', 'PLANIFICADOR']
    if any(p in v for p in prof):
        return 'PROFESIONAL'

    return 'OTRO'

df['OCUPACION_MADRE'] = df['OCUPACION_MADRE'].apply(clasificar_ocupacion)
print(df['OCUPACION_MADRE'].value_counts())

Clasificar EDUCACION_MADRE

In [ ]:
def clasificar_educacion(valor):
    if pd.isna(valor):
        return 'NO SE MENCIONA'
    v = str(valor).upper().strip()
    if v in ['NINGUN NIVEL/ILETRADO', 'INCIAL/PRE-ESCOLAR']:
        return 'SIN EDUCACION'
    elif v in ['PRIMARIA INCOMPLETA', 'PRIMARIA COMPLETA']:
        return 'PRIMARIA'
    elif v in ['SECUNDARIA INCOMPLETA', 'SECUNDARIA COMPLETA']:
        return 'SECUNDARIA'
    elif v in ['SUPERIOR NO UNIV. INCOMPLETA', 'SUPERIOR NO UNIV. COMPLETA',
               'SUPERIOR UNIV. INCOMPLETA', 'SUPERIOR UNIV. COMPLETA']:
        return 'SUPERIOR'
    elif v == 'NO SE MENCIONA':
        return 'NO SE MENCIONA'
    else:
        return 'OTRO'

df['EDUCACION_MADRE'] = df['EDUCACION_MADRE'].apply(clasificar_educacion)
print(df['EDUCACION_MADRE'].value_counts())

Eliminar datos imposibles y limpiar PAIS_MADRE

In [ ]:
filas_antes = len(df)
df = df[df['TALLA_NACIDO'] <= 60]
df = df[df['PESO_NACIDO'] <= 6000]
df['PAIS_MADRE'] = df['PAIS_MADRE'].str.strip()

print(f"Registros eliminados por extremos: {filas_antes - len(df)}")
print(f"Total final: {len(df)}")
print(f"Nulos totales: {df.isnull().sum().sum()}")

Guardar dataset final

In [ ]:
from google.colab import files

ruta_final = '/content/CNV_CLUSTERING_READY.csv'
df.to_csv(ruta_final, index=False, encoding='utf-8-sig')
print(f"Dataset final guardado: {len(df)} registros, {len(df.columns)} columnas.")
files.download(ruta_final)

In [3]:
import pandas as pd

ruta_final = '/content/drive/MyDrive/IA/BIRTH/CNV_FINAL_MODIFICADO-JUNIO.csv'
df = pd.read_csv(ruta_final)

selected_columns = [
    'PESO_NACIDO', 'TALLA_NACIDO', 'SEM_EMB', 'EDAD_MADRE', 'EMB_MADRE',
    'CONDIC_PARTO', 'TIPO_PARTO', 'ESTADO_CIVIL', 'EDUCACION_MADRE', 'OCUPACION_MADRE',
    'ABORTOS_MADRE', 'LUGAR_NACIDO', 'ATIENDE_PARTO', 'FINANCIADOR_PARTO'
]

df_filtered = df[selected_columns]
df_filtered.head(10)

,PESO_NACIDO,TALLA_NACIDO,SEM_EMB,EDAD_MADRE,EMB_MADRE,CONDIC_PARTO,TIPO_PARTO,ESTADO_CIVIL,EDUCACION_MADRE,OCUPACION_MADRE,ABORTOS_MADRE,LUGAR_NACIDO,ATIENDE_PARTO,FINANCIADOR_PARTO
0,3620,52.0,40,34,3.0,EUTOCICO,UNICO,SOLTERO,SECUNDARIA,AMA DE CASA,SI,ESTABLECIMIENTO DE SALUD,OBSTETRA,SIS
1,3005,50.0,37,27,3.0,CESAREA,UNICO,SOLTERO,SUPERIOR,AMA DE CASA,NO,ESTABLECIMIENTO DE SALUD,MEDICO GINECO-OBSTETRA,SIS
2,3055,49.0,37,35,1.0,CESAREA,UNICO,SOLTERO,SUPERIOR,PROFESIONAL,NO,ESTABLECIMIENTO DE SALUD,MEDICO GINECO-OBSTETRA,PRIVADOS
3,3500,50.0,38,33,3.0,EUTOCICO,UNICO,SOLTERO,SUPERIOR,PROFESIONAL,NO,ESTABLECIMIENTO DE SALUD,OBSTETRA,ESSALUD
4,3430,50.1,39,25,3.0,EUTOCICO,UNICO,SOLTERO,SECUNDARIA,AMA DE CASA,NO,ESTABLECIMIENTO DE SALUD,OBSTETRA,SIS
5,2930,47.0,37,23,1.0,EUTOCICO,UNICO,CASADO,SECUNDARIA,AMA DE CASA,NO,ESTABLECIMIENTO DE SALUD,OBSTETRA,SIS
6,3300,49.3,37,35,3.0,EUTOCICO,UNICO,SOLTERO,SECUNDARIA,AMA DE CASA,NO,ESTABLECIMIENTO DE SALUD,OBSTETRA,SIS
7,2700,47.0,39,32,4.0,EUTOCICO,UNICO,SOLTERO,PRIMARIA,AMA DE CASA,NO,DOMICILIO,FAMILIAR,SIS
8,3200,48.2,39,30,2.0,EUTOCICO,UNICO,SOLTERO,PRIMARIA,AMA DE CASA,NO,ESTABLECIMIENTO DE SALUD,OBSTETRA,SIS
9,2830,48.4,39,18,1.0,EUTOCICO,UNICO,NO SE MENCIONA,SECUNDARIA,AMA DE CASA,NO,ESTABLECIMIENTO DE SALUD,OBSTETRA,SIS
